# 🎙️ Advanced AI Voice Studio
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
**Zero-Shot Voice Cloning + Speech-to-Speech** using F5-TTS & OpenAI Whisper.
Fully optimised for **Google Colab T4** and **Kaggle T4**.

## 🛠️ Step 1: Environment Setup

In [ ]:
import os
try:
    import google.colab
    ENV = "Colab"
except ImportError:
    ENV = "Kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "Local"
print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Ensure 'Internet' is ON in Kaggle Notebook Settings.")
!pip install -q -U f5-tts openai-whisper gradio torchaudio accelerate bitsandbytes soundfile
!apt-get install -qq -y ffmpeg
print("✅ All dependencies installed.")

## 🧠 Step 2: Load Voice & ASR Models

In [ ]:
import torch, gc, os, soundfile as sf
import gradio as gr
import whisper
from f5_tts.api import F5TTS

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = None
asr_model = None

def clear_cache():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_tts():
    global tts_model
    if tts_model is None:
        print("⏳ Loading F5-TTS...")
        tts_model = F5TTS(device=DEVICE)
        print("✅ F5-TTS ready.")
    return tts_model

def load_asr():
    global asr_model
    if asr_model is None:
        print("⏳ Loading Whisper (base)...")
        # Whisper runs on GPU or CPU; device_map handled internally
        asr_model = whisper.load_model("base", device=DEVICE)
        print("✅ Whisper ready.")
    return asr_model

print("✅ Loaders defined.")

## 🎛️ Step 3: Synthesis & Cloning Logic

In [ ]:
def studio_tts(ref_audio, gen_text, speed):
    if ref_audio is None: return None, "❌ Upload a reference audio clip."
    if not gen_text.strip(): return None, "❌ Script cannot be empty."
    try:
        model = load_tts()
        clear_cache()
        print("🎙️ Synthesizing...")
        wav, sr, _ = model.infer(
            ref_file=ref_audio,
            ref_text="",
            gen_text=gen_text,
            speed=float(speed)
        )
        out = "tts_output.wav"
        sf.write(out, wav, sr)
        return out, f"✅ Done! Saved as {out}"
    except Exception as e:
        return None, f"❌ Error: {e}"

def studio_s2s(ref_audio, source_audio):
    if ref_audio is None or source_audio is None:
        return None, "❌ Upload both reference and source audio."
    try:
        asr = load_asr()
        print("🔊 Transcribing source audio...")
        result = asr.transcribe(source_audio)
        transcript = result["text"]
        print(f"  Transcript: {transcript[:80]}...")
        return studio_tts(ref_audio, transcript, 1.0)
    except Exception as e:
        return None, f"❌ Error: {e}"

print("✅ Studio functions ready.")

## 🏗️ Step 4: Voice Dashboard

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="2M AI Voice Studio") as studio:
    gr.Markdown("# 🎙️ 2M Advanced AI Voice Studio\n**F5-TTS Zero-Shot Cloning · Whisper S2S · Colab & Kaggle T4**")

    with gr.Tab("🧬 Zero-Shot Cloning"):
        with gr.Row():
            with gr.Column():
                ref_in    = gr.Audio(label="Reference Voice (5–15 seconds)", type="filepath")
                script_in = gr.Textbox(label="Script to Synthesize", lines=5,
                    placeholder="Type the text you want the cloned voice to speak...")
                speed_sl  = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed")
                tts_btn   = gr.Button("🎙️ Clone & Synthesize", variant="primary")
                tts_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column():
                tts_out = gr.Audio(label="Generated Voice", type="filepath")
                tts_dl  = gr.File(label="📥 Download")

        def run_tts(ref, text, speed):
            path, msg = studio_tts(ref, text, speed)
            return path, path, msg

        tts_btn.click(run_tts, [ref_in, script_in, speed_sl], [tts_out, tts_dl, tts_status])

    with gr.Tab("🔄 Voice Changer (S2S)"):
        with gr.Row():
            with gr.Column():
                s2s_ref  = gr.Audio(label="Target Voice (Reference)", type="filepath")
                s2s_src  = gr.Audio(label="Source Audio (to convert)", type="filepath")
                s2s_btn  = gr.Button("🔄 Swap Voice", variant="primary")
                s2s_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column():
                s2s_out = gr.Audio(label="Converted Output", type="filepath")
                s2s_dl  = gr.File(label="📥 Download")

        def run_s2s(ref, src):
            path, msg = studio_s2s(ref, src)
            return path, path, msg

        s2s_btn.click(run_s2s, [s2s_ref, s2s_src], [s2s_out, s2s_dl, s2s_status])

studio.launch(share=True, debug=False)

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*